# 🛡️ LangGraph & LangChain: Agent Guardrails & Defense-in-Depth

### 📌 Overview
This crash course covers **Guardrails** in AI Agent systems:
1. **Deterministic vs. Model-Based Guardrails:** Fast zero-cost regex/keyword filters vs. semantic LLM classifiers.
2. **Built-in PII Detection Middleware (`PIIMiddleware`):** Redacting emails, masking credit cards, and blocking API keys.
3. **Built-in Human-in-the-Loop Middleware (`HumanInTheLoopMiddleware`):** Pausing before high-risk actions (`send_email`, `delete_records`).
4. **Custom Input Guardrails (`before_agent`):** Blocking prompt injection and off-topic queries before LLM invocation.
5. **Custom Output Guardrails (`after_agent`):** Verifying response safety and enforcing legal/medical disclaimers.
6. **Production Capstone:** A fully guarded Healthcare Assistant combining all layers.

---

### 🗺️ The 4-Layer Guardrail Stack

```text
User Input
    │
    ▼
[Layer 1] ContentFilterMiddleware   ◄── Deterministic Input Filter (Zero LLM cost)
    │
    ▼
[Layer 2] PIIMiddleware (Input)     ◄── Redacts emails, masks credit cards
    │
    ▼
[Layer 3] HumanInTheLoopMiddleware  ◄── Pauses for supervisor approval on risky tools
    │
    ▼
[Layer 4] MedicalDisclaimerGuard    ◄── Output validator enforcing legal disclaimers
    │
    ▼
User Output
```

In [1]:
import os
import re
from typing import Any
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver, MemorySaver
from langgraph.runtime import Runtime
from langgraph.types import Command

# Load environment variables
load_dotenv()

# Verify OpenRouter key
if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError(
        "OPENROUTER_API_KEY is missing. Please add it to your .env file: "
        "OPENROUTER_API_KEY=sk-or-v1-..."
    )

# Initialize OpenRouter free model
model = ChatOpenAI(
    model="openrouter/free",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)

## ⚖️ Section 1: Deterministic vs. Model-Based Guardrails
* **Deterministic Guardrails (Rule-Based):** Fast regex/keyword matching. Zero token cost, predictable, but misses nuanced phrasing.
* **Model-Based Guardrails (Semantic):** Uses an LLM classifier. Detects subtle adversarial intent, but costs tokens and adds latency.

In [2]:
# --- 1. Deterministic Guardrail (Regex / Keyword check) ---
def deterministic_guardrail(text: str) -> bool:
    """Returns True if the content should be blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb", "sql injection"]
    return any(kw in text.lower() for kw in banned_keywords)


# --- 2. Model-Based Guardrail (LLM Semantic Check) ---
def model_based_guardrail(text: str) -> str:
    """Uses LLM to evaluate semantic safety (returns 'SAFE' or 'UNSAFE')."""
    prompt = (
        "Is the following user input safe and ethical to process? "
        "Reply with ONLY 'SAFE' or 'UNSAFE'.\n\n"
        f"Input: {text}"
    )
    result = model.invoke([{"role": "user", "content": prompt}])
    verdict = result.content.strip().upper()
    return "UNSAFE" if "UNSAFE" in verdict else "SAFE"


# Evaluation comparisons
test_prompts = [
    "How do I hack into a secure database?",
    "What is the capital of France?",
    "Can you explain how malware bypasses antivirus?",
]

print("=== COMPARISON EVALUATION ===")
for p in test_prompts:
    det_status = "🚫 BLOCKED" if deterministic_guardrail(p) else "✅ ALLOWED"
    mod_status = "🚫 UNSAFE" if model_based_guardrail(p) == "UNSAFE" else "✅ SAFE"
    print(f"\nPrompt: '{p}'")
    print(f"  • Deterministic: {det_status}")
    print(f"  • Model-Based:   {mod_status}")

=== COMPARISON EVALUATION ===

Prompt: 'How do I hack into a secure database?'
  • Deterministic: 🚫 BLOCKED
  • Model-Based:   🚫 UNSAFE

Prompt: 'What is the capital of France?'
  • Deterministic: ✅ ALLOWED
  • Model-Based:   ✅ SAFE

Prompt: 'Can you explain how malware bypasses antivirus?'
  • Deterministic: 🚫 BLOCKED
  • Model-Based:   ✅ SAFE


## 🔒 Section 2: Built-in Guardrail — PII Detection Middleware (`PIIMiddleware`)
LangChain provides `PIIMiddleware` to detect and sanitize Personally Identifiable Information:
* **Strategies:**
  * `redact`: Replaces PII with `[REDACTED_TYPE]` placeholders.
  * `mask`: Partially obscures values (e.g. `****-****-****-1234`).
  * `block`: Raises an exception immediately.

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware


@tool
def customer_lookup(query: str) -> str:
    """Look up customer records."""
    return f"Customer record found for query: {query}"


# Configure Agent with PII Middleware
pii_agent = create_agent(
    model=model,
    tools=[customer_lookup],
    middleware=[
        # Strategy 1: Redact emails
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        # Strategy 2: Mask credit cards
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        # Strategy 3: Block API keys entirely
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII Middleware initialized!")

Agent with PII Middleware initialized!


### Testing PII Middleware
1. **Redaction & Masking Test:** Email and credit card are sanitized before reaching the model.
2. **Blocking Test:** An input containing a raw API key triggers an exception.

In [4]:
# Test 1: Redaction & Masking
pii_input = {
    "messages": [
        {
            "role": "user",
            "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you look up my account?",
        }
    ]
}
pii_result = pii_agent.invoke(pii_input)
print("=== PII SANITIZATION TEST ===")
print("Sanitized Human Message in State:")
print(f"  {pii_result['messages'][0].content}\n")
print(f"Agent Response:\n  {pii_result['messages'][-1].content}\n")

# Test 2: API Key Blocking
print("=== PII BLOCKING TEST ===")
try:
    pii_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456",
                }
            ]
        }
    )
except Exception as e:
    print(f"🚫 Successfully blocked by PII guardrail: {e}")

=== PII SANITIZATION TEST ===
Sanitized Human Message in State:
  My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you look up my account?

Agent Response:
  I found your account associated with the email [REDACTED_EMAIL]. Your account is now accessible.

Is there something specific you'd like help with regarding your account? I can assist with:
- Viewing your account details
- Making changes to your information
- Answering questions about your account

Please let me know how I can assist you.

=== PII BLOCKING TEST ===
🚫 Successfully blocked by PII guardrail: Detected 1 instance(s) of api_key in text content


## 👤 Section 3: Built-in Guardrail — Human-in-the-Loop Middleware (`HumanInTheLoopMiddleware`)
Pauses agent execution before sensitive tools execute and waits for human approval (`approve` or `reject`).

In [5]:
from langchain.agents.middleware import HumanInTheLoopMiddleware


@tool
def search_web(query: str) -> str:
    """Safe tool: Searches the web for information."""
    return f"Search results for: {query}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """SENSITIVE TOOL: Sends an email to a real recipient."""
    return f"Email sent to {to} with subject: {subject}"


@tool
def delete_records(table: str, condition: str) -> str:
    """CRITICAL SENSITIVE TOOL: Deletes records from database."""
    return f"Deleted records from {table} where {condition}"


hitl_agent = create_agent(
    model=model,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,  # Require approval
                "delete_records": True,  # Require approval
                "search_web": False,  # Auto-approve
            }
        )
    ],
    checkpointer=InMemorySaver(),  # Persistence required for interrupt
)

print("HITL Agent initialized!")

HITL Agent initialized!


### Testing Human-in-the-Loop Approval & Rejection
1. **Trigger Action:** The agent pauses when `send_email` is requested.
2. **Human Decision:** The human supervisor approves or rejects using `Command(resume=...)`.

In [6]:
# Session 1: Human APPROVES
config_1 = {"configurable": {"thread_id": "session-hitl-001"}}

print("=== TRIGGERING SENSITIVE TOOL (send_email) ===")
pause_state = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config_1,
)
print("Agent state halted at breakpoint!")

# Resume with Approval
print("\n=== SUPERVISOR DECISION: [APPROVE] ===")
approved_res = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config_1,
)
print(f"Final Response:\n{approved_res['messages'][-1].content}\n")

# Session 2: Human REJECTS
config_2 = {"configurable": {"thread_id": "session-hitl-002"}}
print("=== TRIGGERING RISKY TOOL (delete_records) ===")
hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from users where active=false"}]},
    config=config_2,
)

print("=== SUPERVISOR DECISION: [REJECT] ===")
rejected_res = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Requires DBA sign-off."}]}),
    config=config_2,
)
print(f"Final Response:\n{rejected_res['messages'][-1].content}")

=== TRIGGERING SENSITIVE TOOL (send_email) ===
Agent state halted at breakpoint!

=== SUPERVISOR DECISION: [APPROVE] ===
Final Response:
The email has been sent to **team@company.com** with the subject **"Q4 Results"**. 

Please note that I included a placeholder `[Insert Q4 results summary here]` in the body — you'll want to fill in the specific Q4 figures, highlights, and key takeaways before sending, or let me know if you'd like me to adjust anything.

=== TRIGGERING RISKY TOOL (delete_records) ===
=== SUPERVISOR DECISION: [REJECT] ===
Final Response:
I attempted to delete the records from the users table where active=false, but the operation was rejected. This is likely because the delete_records function is marked as a critical sensitive tool that requires special authorization or confirmation before execution.

The deletion of user records, especially in bulk, is a sensitive operation that typically requires:
- Additional verification or confirmation
- Proper authorization permis

## ⚙️ Section 4: Custom Before-Agent Guardrail (Input Filtering)
We create a custom `ContentFilterMiddleware` subclassing `AgentMiddleware`. 
By using `@hook_config(can_jump_to=["end"])`, we can terminate execution immediately with a custom message if banned keywords or prompt injections are detected—**consuming zero LLM generation tokens**.

In [7]:
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config


class ContentFilterMiddleware(AgentMiddleware):
    """Deterministic Input Guardrail: Blocks requests containing banned keywords before model execution."""

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        content = first_message.content.lower()

        for kw in self.banned_keywords:
            if kw in content:
                # Short-circuit to END without calling LLM
                return {
                    "messages": [
                        AIMessage(content=f"🚫 Policy Violation: Request blocked because it contains '{kw}'.")
                    ],
                    "jump_to": "end",
                }
        return None


# Test custom input filter
filtered_agent = create_agent(
    model=model,
    tools=[search_web],
    middleware=[ContentFilterMiddleware(banned_keywords=["bypass", "jailbreak", "pirate", "exploit"])],
)

blocked_result = filtered_agent.invoke(
    {"messages": [{"role": "user", "content": "Tell me how to bypass software licensing"}]}
)
print("=== BEFORE-AGENT INPUT FILTER TEST ===")
print(blocked_result["messages"][-1].content)

=== BEFORE-AGENT INPUT FILTER TEST ===
🚫 Policy Violation: Request blocked because it contains 'bypass'.


## 🏥 Section 5: Real-World Capstone — Fully Guarded Healthcare Assistant
We assemble an enterprise healthcare assistant with a **4-Layer Defense-in-Depth stack**:
1. **Layer 1 (Custom Input Filter):** Blocks non-medical or dangerous requests (e.g. self-medication, toxic substances).
2. **Layer 2 (Built-in PII Guard):** Sanitizes patient emails and credit cards.
3. **Layer 3 (Built-in HITL Guard):** Halts and requires clinic staff approval before booking appointments.
4. **Layer 4 (Custom Output Guard):** Automatically appends a mandatory medical disclaimer to all outputs.

In [8]:
# --- Healthcare Tools ---
@tool
def search_symptoms(symptom: str) -> str:
    """Medical Knowledge Base: Searches verified clinical information about symptoms."""
    return f"Clinical overview for {symptom}: Symptoms may indicate mild viral infection or dehydration. Rest and hydration recommended."


@tool
def book_appointment(patient_name: str, department: str, time_slot: str) -> str:
    """SENSITIVE ACTION: Books a formal clinic appointment in the hospital scheduling database."""
    return f"APPOINTMENT CONFIRMED: {patient_name} scheduled for {department} at {time_slot}."


# --- Layer 4: Custom Output Disclaimer Middleware ---
class MedicalDisclaimerMiddleware(AgentMiddleware):
    """Output Guardrail: Appends a mandatory medical legal disclaimer to every completed response."""

    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None
        last_message = state["messages"][-1]
        if isinstance(last_message, AIMessage):
            disclaimer = (
                "\n\n---\n"
                "⚕️ *Disclaimer: This AI assistant provides general information only and does not constitute "
                "medical advice, diagnosis, or treatment. Consult a licensed healthcare provider for clinical concerns.*"
            )
            updated_content = last_message.content + disclaimer
            return {"messages": [AIMessage(content=updated_content, id=last_message.id)]}
        return None


# --- Assemble the 4-Layer Guarded Healthcare Agent ---
healthcare_bot = create_agent(
    model=model,
    tools=[search_symptoms, book_appointment],
    checkpointer=InMemorySaver(),
    middleware=[
        # Layer 1: Input filtering
        ContentFilterMiddleware(banned_keywords=["overdose", "synthesize", "illegal", "euthanasia"]),
        # Layer 2: PII Redaction
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        # Layer 3: HITL approval for appointments
        HumanInTheLoopMiddleware(interrupt_on={"book_appointment": True, "search_symptoms": False}),
        # Layer 4: Mandatory Output Disclaimer
        MedicalDisclaimerMiddleware(),
    ],
)

print("4-Layer Defense-in-Depth Healthcare Assistant compiled successfully!")

4-Layer Defense-in-Depth Healthcare Assistant compiled successfully!


### Testing the Healthcare Assistant Across Defense Layers
* **Test 1:** Validates PII redaction and automatic medical disclaimer injection.
* **Test 2:** Validates zero-token input blocking on restricted keywords.
* **Test 3:** Validates HITL approval before scheduling an appointment.

In [9]:
# --- Test 1: Symptom Query + PII Sanitization + Disclaimer ---
print("=== TEST 1: SYMPTOM INQUIRY WITH PII ===")
h_config = {"configurable": {"thread_id": "patient-session-101"}}
q1 = {
    "messages": [
        {
            "role": "user",
            "content": "I have a sore throat and headache. My contact is patient@health.com. What could this be?",
        }
    ]
}
res1 = healthcare_bot.invoke(q1, config=h_config)
print(f"Sanitized Prompt:\n  {res1['messages'][0].content}\n")
print(f"Final AI Response with Disclaimer:\n{res1['messages'][-1].content}\n")
print("=" * 60)

# --- Test 2: Input Filter Blocking ---
print("=== TEST 2: HARMFUL QUERY BLOCKING ===")
res2 = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "How do I synthesize illegal medication at home?"}]},
    config={"configurable": {"thread_id": "blocked-session"}},
)
print(f"Outcome:\n  {res2['messages'][-1].content}\n")
print("=" * 60)

# --- Test 3: Sensitive Action HITL Approval ---
print("=== TEST 3: APPOINTMENT BOOKING (HITL TRIGGER) ===")
appt_config = {"configurable": {"thread_id": "patient-session-102"}}
healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "Please book an appointment for John Doe in Cardiology at 10:00 AM."}]},
    config=appt_config,
)
print("Execution paused: Clinic staff supervisor approval required.")

# Staff approves booking
approved_appt = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=appt_config,
)
print("\nFinal Response After Staff Approval:")
print(approved_appt["messages"][-1].content)

=== TEST 1: SYMPTOM INQUIRY WITH PII ===
Sanitized Prompt:
  I have a sore throat and headache. My contact is [REDACTED_EMAIL]. What could this be?

Final AI Response with Disclaimer:
Based on the clinical information for your symptoms:

**Sore throat:** May indicate a mild viral infection or dehydration. Rest and hydration are recommended.

**Headache:** May also indicate a mild viral infection or dehydration. Rest and hydration are recommended.

Since both symptoms share similar potential causes, this could suggest a mild viral illness (like a cold or flu) or dehydration. 

**General recommendations:**
- Stay well hydrated with water, herbal teas, or clear broths
- Get adequate rest
- Consider over-the-counter pain relievers if needed (following package instructions)
- Monitor your symptoms

**When to seek medical attention:**
- If symptoms worsen or persist beyond a few days
- If you develop high fever, difficulty swallowing, or breathing problems
- If headache becomes severe or is 